# 🧪 Optimized Randomized Search - MAX HISTORY (1999-2026)
### Version Historique Complet (27 ans)
- **Périodes Clés** : Bulle Internet (2000), Subprimes (2008), COVID (2020), Hausse des taux (2022).
- **Réalisme** : Frais (0.1%), Marge (5%), Cash Yield (3%).
- **Filtrage** : Double SMA Actions & ETFs + ADX + ATR%.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import mlflow
import itertools
import random
import warnings
from datetime import datetime
from joblib import Parallel, delayed
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Setup Paths
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())
load_dotenv()

try:
    from src.common.setup_spark import create_spark_session
    from config.config_spark import Paths
    print("✅ Environnement initialisé.")
except Exception as e:
    print(f"❌ Erreur imports : {e}")

/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environnement initialisé.


In [2]:
# 1. CHARGEMENT DES DONNÉES (MAX HISTORY)
def load_all_data():
    spark = create_spark_session("GridSearch_MaxHistory")
    try:
        # S&P 500 & ETFs (Start 1999)
        print("📡 Téléchargement S&P 500 et ETFs depuis 1999...")
        sp500_raw = yf.download('^GSPC', start="1999-01-01", progress=False)
        sp500_raw.index = pd.to_datetime(sp500_raw.index).tz_localize(None).normalize()
        
        etf_tickers = ['XLP', 'XLV', 'XLU', 'XLE', 'XLK', 'XLC', 'XLI', 'XLY', 'XLB', 'XLRE', 'TLT', 'IEF', 'HYG', 'GLD', 'VNQ']
        etfs_raw = yf.download(etf_tickers, start="1999-01-01", progress=False)
        etfs_raw.index = pd.to_datetime(etfs_raw.index).tz_localize(None).normalize()
        
        # Stocks Data Lake (Filtre 1999)
        print("📡 Lecture Delta Lake (Filtre 1999)...")
        df_bronze = spark.read.format("delta").load(Paths.SP500_STOCK_PRICES)
        stocks_raw = df_bronze.filter("date >= '1999-01-01'").select('date', 'symbol', 'adjHigh', 'adjLow', 'adjClose').toPandas()
        stocks_raw['date'] = pd.to_datetime(stocks_raw['date']).dt.tz_localize(None).dt.normalize()
        
        # Pivoting
        print("⚡ Pivotement des données...")
        s_close = stocks_raw.pivot(index='date', columns='symbol', values='adjClose').ffill()
        s_high = stocks_raw.pivot(index='date', columns='symbol', values='adjHigh').ffill()
        s_low = stocks_raw.pivot(index='date', columns='symbol', values='adjLow').ffill()
        
    finally:
        spark.stop()
        print("✅ Données chargées (Période: 1999 - Présent).")
    return sp500_raw, etfs_raw, s_close, s_high, s_low

sp500_data, etf_data, stocks_close, stocks_high, stocks_low = load_all_data()

2026-04-04 11:14:11.203 | INFO     | src.common.setup_spark:create_spark_session:22 - 🛠️ Configurant Spark avec GCS et BigQuery Jars...
26/04/04 11:14:11 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 192.168.1.1 instead (on interface en0)
26/04/04 11:14:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-34111205-bf95-4c34-b597-279af08a7e88;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 66ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default

:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/04 11:14:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/04 11:14:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2026-04-04 11:14:15.886 | SUCCESS  | src.common.setup_spark:create_spark_session:43 - ✅ Spark Session 'GridSearch_MaxHistory' créée avec succès ! (Version: 3.5.3)


📡 Téléchargement S&P 500 et ETFs depuis 1999...
📡 Lecture Delta Lake (Filtre 1999)...


⚡ Pivotement des données...
✅ Données chargées (Période: 1999 - Présent).


In [3]:
def calculate_metrics(returns_series):
    metrics_keys = ['cagr', 'sharpe', 'sortino', 'calmar', 'max_drawdown', 'volatility']
    if returns_series.empty or returns_series.isna().all():
        return {k: 0.0 for k in metrics_keys}
    
    rets = returns_series.fillna(0.0).clip(-0.5, 0.5)
    cum_rets = (1 + rets).cumprod()
    ann = 52
    yrs = max(len(rets) / ann, 0.1)
    
    final_nav = cum_rets.iloc[-1] if not cum_rets.empty else 1.0
    cagr = (final_nav)**(1/yrs) - 1 if final_nav > 0 else -1
    
    vol = rets.std() * np.sqrt(ann)
    sharpe = (rets.mean() * ann) / vol if vol > 1e-6 else 0
    
    down_rets = rets[rets < 0]
    down_vol = (down_rets.std() * np.sqrt(ann)) if not down_rets.empty else 1e-6
    sortino = (rets.mean() * ann) / down_vol if down_vol > 1e-6 else 0
    
    m_max = cum_rets.cummax()
    dd = (cum_rets - m_max) / m_max
    m_dd = dd.min() if not dd.empty else -1
    calmar = cagr / abs(m_dd) if abs(m_dd) > 1e-6 else 0
    
    return {'cagr': float(cagr), 'sharpe': float(sharpe), 'sortino': float(sortino), 'calmar': float(calmar), 'max_drawdown': float(m_dd), 'volatility': float(vol)}

In [4]:
class RegimeSwitchingBacktester:
    def __init__(self, config, sp500_raw, etfs_raw, s_close, s_high, s_low):
        self.config = config
        self.sp500_raw = sp500_raw
        self.etfs_raw = etfs_raw
        self.s_close = s_close
        self.s_high = s_high
        self.s_low = s_low
        
        self.fees = config.get('fees', 0.001)
        self.m_rate = config.get('margin_rate', 0.06)
        self.c_yield = config.get('cash_yield', 0.02)

    def run(self):
        # 1. REGIME SP500
        sp_p = self.sp500_raw.xs('Close', 1, 0).iloc[:,0] if isinstance(self.sp500_raw.columns, pd.MultiIndex) else self.sp500_raw.iloc[:,0]
        sp500 = pd.DataFrame(sp_p.resample('W-FRI').last().ffill())
        sp500.columns = ['Price']
        reg_f, reg_s = self.config['sp500_sma_fast'], self.config['sp500_sma_slow']
        
        sma_f = ta.trend.sma_indicator(sp500['Price'], reg_f)
        sma_s = ta.trend.sma_indicator(sp500['Price'], reg_s)
        cond_bull = ((sma_f > sma_s) & (sp500['Price'] > sma_s)) | ((sma_f < sma_s) & (sp500['Price'] > sma_f))
        sp500['Regime'] = np.where(cond_bull, 'Bull', 'Bear')
        
        # 2. ETFs
        e_close = self.etfs_raw.xs('Close', 1, 0) if isinstance(self.etfs_raw.columns, pd.MultiIndex) else self.etfs_raw
        e_close = e_close.resample('W-FRI').last().ffill()
        e_elig = pd.DataFrame(False, index=e_close.index, columns=e_close.columns)
        e_mom = e_close.pct_change(self.config['etf_mom_period'])
        
        e_ma_f = e_close.apply(lambda x: ta.trend.sma_indicator(x, self.config['etf_sma_fast']))
        e_ma_s = e_close.apply(lambda x: ta.trend.sma_indicator(x, self.config['etf_sma_slow']))
        
        for t in e_close.columns:
            e_elig[t] = (e_ma_f[t] > e_ma_s[t]) & (e_close[t] > e_ma_s[t])
            
        # 3. STOCKS
        sc = self.s_close.resample('W-FRI').last().ffill()
        sh = self.s_high.resample('W-FRI').last().ffill()
        sl = self.s_low.resample('W-FRI').last().ffill()
        
        s_elig = pd.DataFrame(False, index=sc.index, columns=sc.columns)
        s_mom = sc.pct_change(self.config['stock_mom_period'])
        
        # 👉 L'indicateur Momentum 1W pour ta condition
        s_mom_1w = sc.pct_change(1) 
        
        s_ma_f = sc.apply(lambda x: ta.trend.sma_indicator(x, self.config['stock_sma_fast']))
        s_ma_s = sc.apply(lambda x: ta.trend.sma_indicator(x, self.config['stock_sma_slow']))
        
        for t in sc.columns:
            try:
                adx = ta.trend.adx(sh[t], sl[t], sc[t], 20, fillna=True)
                atr = ta.volatility.average_true_range(sh[t], sl[t], sc[t], 14, fillna=True)
                
                cond_trend = (s_ma_f[t] > s_ma_s[t]) & (sc[t] > s_ma_s[t])
                cond_strength = adx > self.config['stock_adx_threshold']
                cond_volatility = (atr / sc[t]) < self.config['stock_atr_threshold']
                
                # 👉 TA CONDITION EXACTE RENDUE DYNAMIQUE
                if self.config.get('use_cond_1W', True):
                    cond_1W = (s_mom_1w[t] < 0.0) 
                else:
                    cond_1W = True # Si False dans la grille, la condition est ignorée (toujours Vraie)
                
                # Assemblage des filtres
                s_elig[t] = cond_trend & cond_strength & cond_volatility & cond_1W
                
            except: continue

        # 4. SIMULATION
        cdates = sorted(sp500.index.intersection(e_close.index).intersection(sc.index))
        rebal_freq = self.config['rebalance_freq']
        rebal_dates = set(pd.Series(cdates).groupby(pd.to_datetime(cdates).to_period(rebal_freq)).max().dt.strftime('%Y-%m-%d'))
        
        port, rets = [], []
        lev = self.config['leverage']
        
        for i, d in enumerate(cdates):
            # --- A. Calcul des performances de la semaine ---
            if i > 0:
                w_ret = 0.0
                total_invested = 0.0
                if port:
                    pd_d = cdates[i-1]
                    for pos in port:
                        source = sc if pos['Type'] == 'Stock' else e_close
                        try:
                            r = (source.at[d, pos['Ticker']] / source.at[pd_d, pos['Ticker']]) - 1
                            if not np.isnan(r): 
                                w_ret += r * pos['Weight']
                                total_invested += pos['Weight']
                        except: pass
                
                # Coûts fixes (Cash yield si < 100%, Margin rate si > 100%)
                cash_weight = max(0, 1.0 - total_invested)
                borrowed_weight = max(0, total_invested - 1.0)
                
                w_ret += cash_weight * (self.c_yield / 52)
                w_ret -= borrowed_weight * (self.m_rate / 52)
                
                rets.append(w_ret)
            
            d_str = d.strftime('%Y-%m-%d')
            
            # --- B. STOP-LOSS HEBDOMADAIRE (Vérifié toutes les semaines) ---
            surviving_portfolio = []
            for pos in port:
                kept = True
                if pos['Type'] == 'Stock':
                    # Si le prix passe sous la SMA rapide (ex: SMA 26), on coupe
                    if sc.at[d, pos['Ticker']] < s_ma_f.at[d, pos['Ticker']]:
                        kept = False
                else:
                    # Pour l'ETF, si le prix passe sous la SMA lente (ex: SMA 50), on coupe
                    if e_close.at[d, pos['Ticker']] < e_ma_s.at[d, pos['Ticker']]:
                        kept = False
                
                if kept: surviving_portfolio.append(pos)
            
            port = surviving_portfolio

            # --- C. REBALANCEMENT MENSUEL ---
            if d_str in rebal_dates:
                old_p = {p['Ticker']: p['Weight'] for p in port}
                regime = sp500.at[d, 'Regime']
                
                if regime == 'Bull':
                    # On garde uniquement les actions
                    port = [p for p in port if p['Type'] == 'Stock']
                    cur_t = [p['Ticker'] for p in port]
                    
                    el = s_elig.loc[d]
                    eligible_tickers = el[el].index
                    ranked = s_mom.loc[d, eligible_tickers].sort_values(ascending=False)
                    
                    # Rétention via Buffer (Top 15)
                    kept_tickers = [t for t in cur_t if t in ranked.index[:self.config['buffer_n']]]
                    
                    # Remplissage jusqu'au Top N
                    places_libres = self.config['top_n'] - len(kept_tickers)
                    if places_libres > 0:
                        cand = [t for t in ranked.index if t not in kept_tickers]
                        kept_tickers += cand[:places_libres]
                        
                    # Ajustement dynamique du poids
                    w = lev / len(kept_tickers) if kept_tickers else 0
                    new_p = [{'Ticker': t, 'Type': 'Stock', 'Weight': w} for t in kept_tickers]
                    
                else:
                    # Régime Bear : Top 2 ETFs
                    port = [p for p in port if p['Type'] == 'ETF']
                    cur_t = [p['Ticker'] for p in port]
                    
                    places_libres = 2 - len(port)
                    if places_libres > 0:
                        el = e_elig.loc[d]
                        cand = [t for t in el[el].index if t not in cur_t]
                        if cand:
                            top_e = e_mom.loc[d, cand].nlargest(places_libres).index.tolist()
                            cur_t += top_e
                            
                    # Pas de levier en Bear (Poids max total = 1.0)
                    w = 1.0 / len(cur_t) if cur_t else 0
                    new_p = [{'Ticker': t, 'Type': 'ETF', 'Weight': w} for t in cur_t]
                
                # Frais de turnover
                new_p_dict = {p['Ticker']: p['Weight'] for p in new_p}
                turnover = sum(abs(new_p_dict.get(t,0) - old_p.get(t,0)) for t in (set(old_p) | set(new_p_dict)))
                if rets: rets[-1] -= turnover * self.fees
                
                port = new_p

        return pd.Series(rets, index=cdates[1:])

In [5]:
# 5. SEARCH RANDOM FINAL - MAX HISTORY
def run_experiment(idx, config, sp500, etfs, s_close, s_high, s_low, exp_name):
    mlflow.set_experiment(exp_name)
    with mlflow.start_run(run_name=f"Iter_{idx}"):
        mlflow.log_params(config)
        try:
            tester = RegimeSwitchingBacktester(config, sp500, etfs, s_close, s_high, s_low)
            rets = tester.run()
            mtrs = calculate_metrics(rets)
            mlflow.log_metrics(mtrs)
        except Exception as e: print(f"❌ Iter {idx} fail: {e}")

param_grid = {
    'sp500_sma_fast': [12, 26], 'sp500_sma_slow': [50, 70],
    'etf_sma_fast': [12, 26], 'etf_sma_slow': [26, 50],
    'etf_mom_period': [5, 13, 26, 50], 'stock_mom_period': [5, 13, 26, 50],
    'stock_sma_fast': [12, 26], 'stock_sma_slow': [26, 50],
    'stock_adx_threshold': [10, 20, 30], 'stock_atr_threshold': [0.05, 0.1, 0.2, 0.3],
    'leverage': [1.0, 1.5, 2.0, 2.5], 'rebalance_freq': ['W', 'M', 'Q', '6M'],
    'top_n': [5, 10, 15], 'buffer_n': [10, 15, 20, 25], 
    'fees': [0.001], 'margin_rate': [0.05], 'cash_yield': [0.03],
    'use_pullback': [True, False],
    'use_cond_1W': [True, False], 
}

# Calcule dynamiquement le nombre de combinaisons
total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"🧮 Nombre total de combinaisons possibles : {total_combinations}")

🧮 Nombre total de combinaisons possibles : 9437184


In [ ]:
N_ITER = 100
EXPERIMENT_NAME = "Regime_Momentum_Max_History_v3"
all_configs = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
random.seed(42)
selected = random.sample(all_configs, min(N_ITER, len(all_configs)))

print(f"🚀 Lancement du Search Final sur 27 ANS pour {len(selected)} itérations...")
for i, cfg in enumerate(selected):
    if i % 10 == 0: print(f"Progress: {i}/{len(selected)}")
    run_experiment(i, cfg, sp500_data, etf_data, stocks_close, stocks_high, stocks_low, EXPERIMENT_NAME)
print("✅ Terminé.")